<a href="https://colab.research.google.com/github/gandersen-ctb/skilljar/blob/main/Prompt_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Promp Eval workflow

In [1]:
# install deps.
%pip install anthropic python-dotenv

In [2]:
from shared import max_tokens, add_role_message, chat, run_eval

In [3]:
import json

def generate_test_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
    each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
        {
            "task": "Description of task",
            "format": "json" or "python" or "regex"
        },
        ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
    """

    messages = []
    add_role_message(messages, 'user', prompt)
    add_role_message(messages, 'assistant', "```json")
    model = "claude-haiku-4-5"
    text = chat(messages, model, stop_sequences=["```"])

    return json.loads(text)

In [4]:
dataset = generate_test_dataset()
dataset

[{'task': "Parse an AWS S3 bucket name from an S3 URI (e.g., 's3://my-bucket/path/to/file.txt') and extract just the bucket name",
  'format': 'regex'},
 {'task': "Create a JSON configuration object for an AWS Lambda function that specifies runtime as 'python3.11', memory as 256 MB, timeout as 30 seconds, and environment variables for DB_HOST and API_KEY",
  'format': 'json'},
 {'task': 'Write a Python function that takes an AWS CloudWatch log timestamp in ISO 8601 format and converts it to a Unix timestamp (seconds since epoch)',
  'format': 'python'}]

# Evaluate the dateset

In [5]:
results = run_eval(dataset);
print(json.dumps(results, indent=2))

Model eval score: 7
Code eval score: 10
Model eval score: 7
Code eval score: 10
Model eval score: 7
Code eval score: 10
Average score: 8.5
[
  {
    "output": "\nimport re\n\ndef extract_s3_bucket(uri):\n    match = re.match(r's3://([^/]+)', uri)\n    return match.group(1) if match else None\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name from an S3 URI (e.g., 's3://my-bucket/path/to/file.txt') and extract just the bucket name",
      "format": "regex"
    },
    "score": 8.5,
    "reasoning": "The solution effectively solves the core task with clean, readable code. However, it lacks robustness for edge cases and real-world AWS scenarios. Adding simple null-checking and optionally validating bucket name constraints would significantly improve reliability. The regex pattern itself is correct and efficient for standard S3 URI parsing."
  },
  {
    "output": "\n{\n  \"Runtime\": \"python3.11\",\n  \"MemorySize\": 256,\n  \"Timeout\": 30,\n  \"Environment\": {\n    \"Var